# Обучение моделей cat/dog — оркестратор для ВМ

Запускать **из корня проекта** на ВМ. Каждая ячейка зовёт CLI-шаги из README.

## Подготовка (один раз, в терминале ВМ)

```bash
python3.10 -m venv ~/practicum_venv
source ~/practicum_venv/bin/activate
cd ~/nn_cv_sprint2_full
bash practicum_work/setup_vm.sh
clearml-init
```

Затем откройте этот ноутбук в **VSCode Remote-SSH** и выберите **kernel = `~/practicum_venv`**.

## Что внутри ноутбука
1. sanity-check конфига бейзлайна;
2. обучение бейзлайна — DeepLabV3+ R50-d8;
3. анализ качества лучшего чекпойнта на test.

UNet «с нуля» по обоснованию из учебника (см. README Этап 2) сознательно
не запускается. Эксперименты Этапа 3 (аугментации, лосс 1:3, R101-mg124)
добавляются ниже после получения метрики бейзлайна.

In [1]:
# cwd ядра ставим в корень проекта (VSCode по умолчанию открывает с cwd = папка ноутбука)
%cd ~/nn_cv_sprint2_full

/home/ubuntu/nn_cv_sprint2_full


/home/ubuntu/nn_cv_sprint2_full/practicum_venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})
/home/ubuntu/nn_cv_sprint2_full/practicum_venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


## 1. Sanity-check конфига бейзлайна (без обучения)

In [2]:
!python practicum_work/sanity_check.py configs/deeplabv3plus_practice/deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256.py

[1/6] config loaded: configs/deeplabv3plus_practice/deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256.py
[2/6] train dataset: PracticeDataset  len=179
[3/6] train.txt match: ok (179 samples)
[4/6] val dataset:   PracticeDataset  len=120
/home/ubuntu/nn_cv_sprint2_full/mmseg/models/backbones/resnet.py:431: UserWarning: DeprecationWarning: pretrained is a deprecated, please use "init_cfg" instead
  warnings.warn('DeprecationWarning: pretrained is a deprecated, '
/home/ubuntu/nn_cv_sprint2_full/mmseg/models/losses/cross_entropy_loss.py:250: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  warnings.warn(
[5/6] model forward: ok  (img (1, 3, 256, 256) -> feats [(1, 256, 64, 64), (1, 512, 32, 32), (1, 1024, 32, 32), (1, 2048, 32, 32)])
[6/6] visualizer backends (из конфига): ['LocalVisBackend', 'ClearMLVisBackend']
   

## 2. Обучение бейзлайна — DeepLabV3+ R50-d8

Ожидаемое время — **~30–45 мин на одном T4**: из таблицы замеров скорости в
уроке «Современные модификации» для `deeplabv3plus_r50-d8` это ~0.077 c/итер,
у нас ~1200 итер (100 эпох × ~12 итер/эпоху при batch=16).

Целевая метрика проекта — **mDice > 0.75** на test. ClearML-ссылка
появится в первых строках вывода ниже.

In [3]:
!python tools/train.py configs/deeplabv3plus_practice/deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256.py

05/31 08:13:58 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.12 (main, Aug 15 2025, 14:32:43) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1631462783
    GPU 0: Tesla T4
    CUDA_HOME: /usr
    NVCC: Cuda compilation tools, release 12.2, V12.2.140
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
    PyTorch: 2.0.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash 6dbeffbae1f23cbbeae17adb7b5b13f1f37c080e)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=compute_37,code=sm_37;-genc

## 3. Per-image Dice анализ лучшего чекпойнта бейзлайна на test

Скрипт сохраняет CSV `per_image_dice.csv` плюс по 5 best/worst PNG-триплетов
(image | GT | pred) для отчёта Этапа 2.

In [5]:
import glob

CFG  = "configs/deeplabv3plus_practice/deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256.py"
NAME = CFG.split("/")[-1].replace(".py", "")
ckpts = sorted(glob.glob(f"work_dirs/{NAME}/best_mDice_epoch_*.pth"))
assert ckpts, f"no best checkpoint for {NAME}"
ckpt = ckpts[-1]
print("==>", NAME, ckpt)

!python practicum_work/src/analysis/per_image_dice.py \
    --config {CFG} \
    --checkpoint {ckpt} \
    --split test \
    --out practicum_work/supplementary/viz/test_baseline_deeplab --n 5

==> deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256 work_dirs/deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256/best_mDice_epoch_95.pth


split=test, n=120
/home/ubuntu/nn_cv_sprint2_full/mmseg/models/backbones/resnet.py:431: UserWarning: DeprecationWarning: pretrained is a deprecated, please use "init_cfg" instead
  warnings.warn('DeprecationWarning: pretrained is a deprecated, '
/home/ubuntu/nn_cv_sprint2_full/mmseg/models/losses/cross_entropy_loss.py:250: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  warnings.warn(
Loads checkpoint by local backend from path: work_dirs/deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256/best_mDice_epoch_95.pth
mDice over split: mean=0.8492  median=0.9401
saved 5 worst examples -> practicum_work/supplementary/viz/test_baseline_deeplab/worst
saved 5 best examples -> practicum_work/supplementary/viz/test_baseline_deeplab/best
